In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import json

print("STEP 1: Generating stock data...")

dates = pd.date_range(start='2026-01-01', end='2026-06-24', freq='D')
companies = ['AAPL', 'GOOGL', 'MSFT', 'AMZN', 'TSLA']

data = {
    'date': np.repeat(dates, len(companies)),
    'company': companies * len(dates),
    'open': np.random.uniform(100, 500, len(dates) * len(companies)),
    'high': np.random.uniform(100, 500, len(dates) * len(companies)),
    'low': np.random.uniform(100, 500, len(dates) * len(companies)),
    'close': np.random.uniform(100, 500, len(dates) * len(companies)),
    'volume': np.random.randint(1000000, 10000000, len(dates) * len(companies))
}

df_raw = pd.DataFrame(data)
print(f"Generated {len(df_raw)} records")
print(df_raw.head())

print("\nSTEP 2: Cleaning and validating data...")

null_count = df_raw.isnull().sum().sum()
print(f"Null values found: {null_count}")

df_clean = df_raw.drop_duplicates(subset=['date', 'company'])
print(f"After deduplication: {len(df_clean)} records")

df_clean['date'] = pd.to_datetime(df_clean['date'])
df_clean['open'] = pd.to_numeric(df_clean['open'], errors='coerce')
df_clean['high'] = pd.to_numeric(df_clean['high'], errors='coerce')
df_clean['low'] = pd.to_numeric(df_clean['low'], errors='coerce')
df_clean['close'] = pd.to_numeric(df_clean['close'], errors='coerce')
df_clean['volume'] = pd.to_numeric(df_clean['volume'], errors='coerce')

print(f"Data validation complete. Final records: {len(df_clean)}")

print("\nSTEP 3: Enriching data with features...")

df_clean = df_clean.sort_values(['company', 'date'])
df_clean['daily_return'] = df_clean.groupby('company')['close'].pct_change() * 100

df_clean['ma_5'] = df_clean.groupby('company')['close'].transform(lambda x: x.rolling(window=5).mean())

df_clean['price_range'] = df_clean['high'] - df_clean['low']

df_clean['year'] = df_clean['date'].dt.year
df_clean['month'] = df_clean['date'].dt.month
df_clean['day'] = df_clean['date'].dt.day

print("Features added successfully")
print(df_clean[['date', 'company', 'close', 'daily_return', 'ma_5', 'price_range']].head(10))

print("\nSTEP 4: Aggregations and analytics...")

daily_summary = df_clean.groupby(['date', 'company']).agg({
    'close': 'last',
    'volume': 'sum',
    'daily_return': 'first',
    'price_range': 'first'
}).reset_index()

company_stats = df_clean.groupby('company').agg({
    'close': ['mean', 'min', 'max', 'std'],
    'volume': 'mean',
    'daily_return': 'mean'
}).round(2)

print("Daily Summary:")
print(daily_summary.head())

print("\nCompany Statistics:")
print(company_stats)

volatility = df_clean.groupby('company')['daily_return'].std().round(3)
print("\nVolatility by Company:")
print(volatility)

print("\nSTEP 5: Saving results...")

df_clean.to_csv('stock_data_cleaned.csv', index=False)
daily_summary.to_csv('daily_summary.csv', index=False)
company_stats.to_csv('company_statistics.csv')

print("Files saved successfully")
print("- stock_data_cleaned.csv")
print("- daily_summary.csv")
print("- company_statistics.csv")

print("\nSTEP 6: Data Quality Report...")

print(f"Total records processed: {len(df_clean)}")
print(f"Companies analyzed: {df_clean['company'].nunique()}")
print(f"Date range: {df_clean['date'].min()} to {df_clean['date'].max()}")
print(f"Missing values: {df_clean.isnull().sum().sum()}")

print("\nPipeline completed successfully!")

STEP 1: Generating stock data...
Generated 875 records
        date company        open        high         low       close   volume
0 2026-01-01    AAPL  184.428582  254.524357  441.728896  233.680916  1737592
1 2026-01-01   GOOGL  422.497868  160.833453  178.505171  455.986370  6458543
2 2026-01-01    MSFT  216.645846  466.807043  382.596072  447.541713  4625863
3 2026-01-01    AMZN  194.358456  407.948527  457.664513  427.574872  8165430
4 2026-01-01    TSLA  226.642909  330.261357  147.047221  337.204298  3153249

STEP 2: Cleaning and validating data...
Null values found: 0
After deduplication: 875 records
Data validation complete. Final records: 875

STEP 3: Enriching data with features...
Features added successfully
         date company       close  daily_return        ma_5  price_range
0  2026-01-01    AAPL  233.680916           NaN         NaN  -187.204539
5  2026-01-02    AAPL  334.859556     43.297776         NaN    43.567692
10 2026-01-03    AAPL  101.553174    -69.672905  